In [16]:
from src.preprocessing import load_dataset, validate_bundle
from src.splitter import build_splits
from src.trainer import run_optuna, dnn_objective, xgb_objective, rf_objective
from src.evaluation import OOFManager
import optuna.visualization as vis
import joblib

In [3]:
bundle = load_dataset("../data/processed/df.csv")
validate_bundle(bundle)


📊 Preprocessing Validation
Samples        : 3715
Features       : 1036
Descriptors    : 12
Fingerprints   : 1024
Target mean    : 6.7329
Target std     : 1.5795


In [4]:
splitter, outer, inner = build_splits(
    bundle.X,
    bundle.groups,
    outer_splits=5,
    inner_splits=3
)


🧪 Scaffold Split Report
Fold 01 | Train:  3012 (1102 scaf) | Test:   703 ( 276 scaf) | Leakage: 0
Fold 02 | Train:  3098 (1102 scaf) | Test:   617 ( 276 scaf) | Leakage: 0
Fold 03 | Train:  3060 (1102 scaf) | Test:   655 ( 276 scaf) | Leakage: 0
Fold 04 | Train:  3125 (1102 scaf) | Test:   590 ( 276 scaf) | Leakage: 0
Fold 05 | Train:  2995 (1102 scaf) | Test:   720 ( 276 scaf) | Leakage: 0


GroupKFold(n_splits=3, random_state=None, shuffle=False)

In [6]:
study_xgb = run_optuna(xgb_objective, bundle, outer, n_trials=50)

[I 2026-05-05 20:06:00,093] A new study created in memory with name: no-name-6f4c7e6f-6dc8-4db1-bed9-1d02bd31d245
[I 2026-05-05 20:06:12,547] Trial 0 finished with value: 0.5702706821161267 and parameters: {'n_estimators': 689, 'max_depth': 3, 'learning_rate': 0.01244417781270052, 'subsample': 0.8702006831735373, 'colsample_bytree': 0.6315441085656498}. Best is trial 0 with value: 0.5702706821161267.
[I 2026-05-05 20:06:58,343] Trial 1 finished with value: 0.511654571193153 and parameters: {'n_estimators': 901, 'max_depth': 8, 'learning_rate': 0.016270497616170713, 'subsample': 0.7329726373590479, 'colsample_bytree': 0.640783669614634}. Best is trial 1 with value: 0.511654571193153.
[I 2026-05-05 20:07:06,042] Trial 2 finished with value: 0.5527189785221068 and parameters: {'n_estimators': 307, 'max_depth': 3, 'learning_rate': 0.04282380925182784, 'subsample': 0.7826857607239333, 'colsample_bytree': 0.6858356742314595}. Best is trial 1 with value: 0.511654571193153.
[I 2026-05-05 20:07

In [ ]:
vis.plot_optimization_history(study_xgb).show()
vis.plot_param_importances(study_xgb).show()

In [13]:
study_xgb.best_params


{'n_estimators': 866,
 'max_depth': 9,
 'learning_rate': 0.015565114081066417,
 'subsample': 0.7974796938236904,
 'colsample_bytree': 0.6244505448177882}

In [17]:
joblib.dump(study_xgb, "../data/optim/xgb_optuna_study.pkl")

['../data/optim/xgb_optuna_study.pkl']

In [7]:
study_rf  = run_optuna(rf_objective, bundle, outer, n_trials=50)

[I 2026-05-05 20:36:15,702] A new study created in memory with name: no-name-b8a88be6-fb5b-4301-8fba-0ba4ef5f0485
[I 2026-05-05 20:38:46,341] Trial 0 finished with value: 0.5518809342149293 and parameters: {'n_estimators': 408, 'max_depth': 22, 'min_samples_split': 9}. Best is trial 0 with value: 0.5518809342149293.
[I 2026-05-05 20:39:40,241] Trial 1 finished with value: 0.6030522474423047 and parameters: {'n_estimators': 206, 'max_depth': 7, 'min_samples_split': 9}. Best is trial 0 with value: 0.5518809342149293.
[I 2026-05-05 20:45:55,303] Trial 2 finished with value: 0.5482184744169274 and parameters: {'n_estimators': 980, 'max_depth': 21, 'min_samples_split': 5}. Best is trial 2 with value: 0.5482184744169274.
[I 2026-05-05 20:51:28,425] Trial 3 finished with value: 0.5537170548748502 and parameters: {'n_estimators': 990, 'max_depth': 17, 'min_samples_split': 8}. Best is trial 2 with value: 0.5482184744169274.
[I 2026-05-05 20:56:18,775] Trial 4 finished with value: 0.552169454708

In [14]:
study_rf.best_params


{'n_estimators': 887, 'max_depth': 29, 'min_samples_split': 2}

In [18]:
joblib.dump(study_rf, "../data/optim/rf_optuna_study.pkl")

['../data/optim/rf_optuna_study.pkl']

In [ ]:
vis.plot_optimization_history(study_rf).show()
vis.plot_param_importances(study_rf).show()

In [8]:
study_dnn = run_optuna(dnn_objective, bundle, outer, n_trials=50)

[I 2026-05-05 23:59:20,457] A new study created in memory with name: no-name-3cf95a7e-e4d2-45a3-ab28-5ebea802d6de
[I 2026-05-06 00:01:40,496] Trial 0 finished with value: 0.5270819110812089 and parameters: {'lr': 0.0026473654418281926, 'dropout': 0.46711013580272687, 'n_layers': 4, 'h0': 448, 'h1': 512, 'h2': 128, 'h3': 128, 'batch': 128}. Best is trial 0 with value: 0.5270819110812089.
[I 2026-05-06 00:03:02,355] Trial 1 finished with value: 0.5243320734314281 and parameters: {'lr': 0.00026218592393987997, 'dropout': 0.296008249908731, 'n_layers': 2, 'h0': 128, 'h1': 192, 'batch': 64}. Best is trial 1 with value: 0.5243320734314281.
[I 2026-05-06 00:06:26,152] Trial 2 finished with value: 0.6853609614161608 and parameters: {'lr': 1.651819063666695e-05, 'dropout': 0.46750053448210105, 'n_layers': 3, 'h0': 256, 'h1': 128, 'h2': 512, 'batch': 32}. Best is trial 1 with value: 0.5243320734314281.
[I 2026-05-06 00:08:15,942] Trial 3 finished with value: 0.5637870206355868 and parameters: {'

In [15]:
study_dnn.best_params


{'lr': 0.0007391992551303584,
 'dropout': 0.10860685748738372,
 'n_layers': 2,
 'h0': 128,
 'h1': 448,
 'batch': 32}

In [19]:
joblib.dump(study_dnn, "../data/optim/dnn_optuna_study.pkl")

['../data/optim/dnn_optuna_study.pkl']

In [ ]:
vis.plot_optimization_history(study_dnn).show()
vis.plot_param_importances(study_dnn).show()

In [20]:
"""oof = OOFManager(len(bundle.y))

# during training:
oof.update(preds, val_idx)

# final:
metrics = oof.compute_metrics(bundle.y, oof.oof_preds)"""

'oof = OOFManager(len(bundle.y))\n\n# during training:\noof.update(preds, val_idx)\n\n# final:\nmetrics = oof.compute_metrics(bundle.y, oof.oof_preds)'

In [21]:
"""df_eval = oof.get_eval_df(bundle.y, bundle.groups)
scaffold_summary(df_eval)"""

'df_eval = oof.get_eval_df(bundle.y, bundle.groups)\nscaffold_summary(df_eval)'